In [ ]:
# ============================================================
# EJECUCIÓN SECUENCIAL DE NOTEBOOKS CON REGISTRO DE ESTADO
# ============================================================

notebooks <- c(
  "z729_st_01_100931.ipynb",
  "z729_st_02_163687.ipynb",
  "z729_st_03_220022.ipynb",
  "z729_st_04_365305.ipynb",
  "z729_st_05_402372.ipynb",
  "z729_st_06_498440.ipynb",
  "z729_st_07_567294.ipynb",
  "z729_st_08_613037.ipynb",
  "z729_st_09_626029.ipynb",
  "z729_st_10_648983.ipynb",
  "z729_st_11_690506.ipynb",
  "z729_st_12_700001.ipynb",
  "z729_st_13_804528.ipynb",
  "z729_st_14_929304.ipynb"
)

# Archivo de seguimiento. Se crea en la carpeta desde donde corre este notebook.
archivo_log <- "log_ejecucion_notebooks.txt"

# Reinicia el log al comenzar una nueva tanda.
cat(
  "============================================================\n",
  "INICIO DE LA EJECUCIÓN: ", format(Sys.time(), "%Y-%m-%d %H:%M:%S"), "\n",
  "Cantidad de notebooks: ", length(notebooks), "\n",
  "============================================================\n\n",
  file = archivo_log,
  sep = ""
)

# Tabla para conservar el resultado de cada notebook.
resultados <- data.frame(
  notebook = notebooks,
  estado = rep("PENDIENTE", length(notebooks)),
  inicio = rep(NA_character_, length(notebooks)),
  fin = rep(NA_character_, length(notebooks)),
  duracion_min = rep(NA_real_, length(notebooks)),
  stringsAsFactors = FALSE
)

for (i in seq_along(notebooks)) {

  notebook <- notebooks[i]
  inicio <- Sys.time()

  resultados$inicio[i] <- format(inicio, "%Y-%m-%d %H:%M:%S")

  mensaje_inicio <- paste0(
    "\n============================================================\n",
    "[", i, "/", length(notebooks), "] INICIANDO: ", notebook, "\n",
    "Hora: ", resultados$inicio[i], "\n",
    "============================================================\n"
  )

  cat(mensaje_inicio)
  cat(mensaje_inicio, file = archivo_log, append = TRUE)

  # Verificación previa: si el archivo no existe, se registra y se continúa.
  if (!file.exists(notebook)) {

    fin <- Sys.time()
    resultados$estado[i] <- "NO ENCONTRADO"
    resultados$fin[i] <- format(fin, "%Y-%m-%d %H:%M:%S")
    resultados$duracion_min[i] <- round(as.numeric(difftime(fin, inicio, units = "mins")), 2)

    mensaje <- paste0(
      "NO EJECUTADO: el archivo no existe.\n",
      "FINALIZADO: ", notebook, " -> NO ENCONTRADO\n"
    )

    cat(mensaje)
    cat(mensaje, file = archivo_log, append = TRUE)

    next
  }

  salida <- system2(
    "jupyter",
    args = c(
      "nbconvert",
      "--to", "notebook",
      "--execute",
      "--ExecutePreprocessor.timeout=-1",
      "--inplace",
      shQuote(notebook)
    ),
    stdout = TRUE,
    stderr = TRUE
  )

  estado_sistema <- attr(salida, "status")

  if (is.null(estado_sistema)) {
    estado_sistema <- 0
  }

  fin <- Sys.time()

  resultados$fin[i] <- format(fin, "%Y-%m-%d %H:%M:%S")
  resultados$duracion_min[i] <- round(
    as.numeric(difftime(fin, inicio, units = "mins")),
    2
  )

  # Guarda también la salida de Jupyter en el log.
  if (length(salida) > 0) {
    cat(paste(salida, collapse = "\n"), "\n")
    cat(
      paste(salida, collapse = "\n"), "\n",
      file = archivo_log,
      append = TRUE
    )
  }

  if (estado_sistema == 0) {

    resultados$estado[i] <- "OK"

    mensaje <- paste0(
      "\nFINALIZADO: ", notebook, " -> OK",
      " | Duración: ", resultados$duracion_min[i], " min\n"
    )

  } else {

    resultados$estado[i] <- "ERROR"

    mensaje <- paste0(
      "\nFINALIZADO: ", notebook, " -> ERROR",
      " | Código: ", estado_sistema,
      " | Duración: ", resultados$duracion_min[i], " min\n",
      "ATENCIÓN: se continúa con el siguiente notebook.\n"
    )
  }

  cat(mensaje)
  cat(mensaje, file = archivo_log, append = TRUE)
}

# ============================================================
# RESUMEN FINAL
# ============================================================

cat("\n\n============================================================\n")
cat("RESUMEN FINAL\n")
cat("============================================================\n")
print(resultados, row.names = FALSE)

cat("\n============================================================\n")
cat("OK:            ", sum(resultados$estado == "OK"), "\n")
cat("ERROR:         ", sum(resultados$estado == "ERROR"), "\n")
cat("NO ENCONTRADO: ", sum(resultados$estado == "NO ENCONTRADO"), "\n")
cat("TOTAL:         ", nrow(resultados), "\n")
cat("============================================================\n")

# Escribe el mismo resumen al final del archivo de texto.
cat(
  "\n\n============================================================\n",
  "RESUMEN FINAL\n",
  "============================================================\n",
  file = archivo_log,
  append = TRUE,
  sep = ""
)

capture.output(
  print(resultados, row.names = FALSE),
  file = archivo_log,
  append = TRUE
)

cat(
  "\nOK:            ", sum(resultados$estado == "OK"),
  "\nERROR:         ", sum(resultados$estado == "ERROR"),
  "\nNO ENCONTRADO: ", sum(resultados$estado == "NO ENCONTRADO"),
  "\nTOTAL:         ", nrow(resultados),
  "\nFIN:           ", format(Sys.time(), "%Y-%m-%d %H:%M:%S"),
  "\n============================================================\n",
  file = archivo_log,
  append = TRUE,
  sep = ""
)

cat("\nSeguimiento guardado en:", archivo_log, "\n")
